## Using an LLM to Interact with Geospatial Data

This notebook is an example of how to use an LLM to interact with geospatial data. In particular, uses the data from the heatwaves project, which is publicly available and can be found [here](https://github.com/unicef/heat).

The objective is to query the data using natural language and get a response in a format that is easy to understand. This notebooks shows how given a metric, a country, and a decade, the LLM can be used to query the data and get a response.


In [1]:
# import necessary modules
import time
from typing import Literal

import ee
import pycountry
from dotenv import load_dotenv
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph.graph import CompiledGraph
from langgraph.prebuilt import create_react_agent

# Load environment variables (openai key)
load_dotenv(override=True)

True

In [2]:
# Authenticate and Initialize Earth Engine
ee.Authenticate()
ee.Initialize(project="unicef-geospatial")

Utils Functions

In [3]:
def print_stream(stream: list) -> None:
    """Print messages from a stream of LangChain agent responses.
    
    Args:
        stream: A list of dictionaries containing LangChain agent responses, where each
            dictionary has a 'messages' key containing either tuple messages or regular 
            LangChain messages.
    """
    for s in stream:
        message = s["messages"][-1]
        # Handle both tuple messages and regular messages
        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print()

def count_tokens(response: dict) -> dict:
    """Count the total tokens used in a LangChain response.
    
    Args:
        response: A dictionary containing LangChain messages with token usage metadata.

    Returns:
        A dictionary with token counts containing:
            - input_tokens: Number of tokens in the prompt
            - output_tokens: Number of tokens in the completion
            - total_tokens: Total number of tokens used
    """
    input_tokens = 0
    output_tokens = 0
    total_tokens = 0
    for message in response["messages"]:
        # Skip messages without metadata or empty metadata
        if not hasattr(message, "response_metadata") or message.response_metadata=={}:
            continue
        token_usage = message.response_metadata.get("token_usage", {})
        input_tokens += token_usage.get("prompt_tokens", 0)
        output_tokens += token_usage.get("completion_tokens", 0)
        total_tokens += token_usage.get("total_tokens", 0)
    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
    }


def ask_and_log_question(graph: CompiledGraph, question: str) -> None:
    """Ask a question using a LangChain graph and log the response details.
    
    Args:
        graph: A compiled LangChain graph that can process questions
        question: The natural language question to ask the graph
        
    Prints:
        - The stream of responses from the graph
        - Response metadata including token counts and execution time
    """
    start = time.time()
    inputs = {
        "messages": [
            {
                "role": "user",
                "content": question,
            }
        ]
    }
    input_tokens = 0
    output_tokens = 0
    total_tokens = 0
    for response in graph.stream(inputs, stream_mode="values"):
        print_stream([response])
        # Get token usage if available in the response
        token_usage = count_tokens(response)
        input_tokens += token_usage["input_tokens"]
        output_tokens += token_usage["output_tokens"]
        total_tokens += token_usage["total_tokens"]

    end = time.time()

    print("================================== Response Metadata ==================================")
    print(f"Input tokens: {input_tokens}")
    print(f"Output tokens: {output_tokens}")
    print(f"Total tokens: {total_tokens}")
    print(f"Time taken: {end - start} seconds")

def standarize_country_name(country: str) -> str:
    """Standardize a country name to its official form using pycountry.

    Args:
        country: A string representing a country nam.e

    Returns:
        The official country name if found in pycountry database, otherwise returns
        the input string unchanged.

    Examples:
        >>> standarize_country_name("US")
        'United States'
        >>> standarize_country_name("FRA")
        'France'
    """
    try:
        country_obj = (
            pycountry.countries.get(name=country)
            or pycountry.countries.get(alpha_2=country)
            or pycountry.countries.get(alpha_3=country)
        )
        if country_obj:
            return country_obj.name
        else:
            return country
    except KeyError:
        return country

In [4]:
METRICS = Literal["frequency", "duration", "severity", "extreme_high_temp"]
DECADES = Literal["1960s", "1970s", "1980s", "1990s", "2000s", "2010s", "2020s"]
COUNTRY_BOUNDRIES_DATASET = "USDOS/LSIB_SIMPLE/2017"
REDUCERS = Literal["mean", "max", "min"]


@tool
def get_heatwave_metric_for_country(
    metric: METRICS, decade: DECADES, country: str, reducer: REDUCERS = "mean"
) -> dict:
    """Get the value of a heatwave metric for a specific country and decade.

    Args:
        metric: One of 'frequency', 'duration', 'severity', 'extreme_high_temp'
        decade: One of '1960s', '1970s', '1980s', '1990s', '2000s', '2010s', '2020s'
        country: Name of the country
        reducer: The statistical operation to apply across all data in the country:
            - 'mean': Average value across the country (default)
            - 'max': Maximum value found anywhere in the country
            - 'min': Minimum value found anywhere in the country

    Returns:
        The value of the heatwave metric for the specified country and decade.
    """
    country_standarized = standarize_country_name(country)
    if country_standarized != country:
        print(f"Country name '{country}' standarized to '{country_standarized}'")

    # Load the heatwave data from the earth engine project assets
    heatwave_tiff = ee.Image(
        f"projects/unicef-geospatial/assets/heatwaves/{metric}/average_heatwaves_{metric}_{decade}_proj_COG"
    )

    # Load country boundaries
    countries_boundries = ee.FeatureCollection(COUNTRY_BOUNDRIES_DATASET)
    country_boundries = countries_boundries.filter(ee.Filter.eq("country_na", country_standarized))

    # Clip the heatwave data to the country boundaries
    country_heatwave = heatwave_tiff.clip(country_boundries)

    # Reduce the heatwave data to the country boundaries
    stats = country_heatwave.reduceRegion(
        reducer=getattr(ee.Reducer, reducer)(),
        geometry=country_boundries.geometry(),
        scale=1000,
        maxPixels=1e13,
    )
    return round(stats.getInfo()["b1"], 3)

In [5]:
# Initialize LLM and create agent
llm = ChatOpenAI(temperature=0.0, model="gpt-4o-mini")
tools = [get_heatwave_metric_for_country]

system_prompt = """You are an expert in analyzing heatwave data across different countries and time periods.

You have access to a tool that can retrieve the value of a heatwave metric for a specific country and decade.

Available metrics:
• frequency: How often heatwaves occur
• duration: Average length of heatwave event (Number of days)
• severity: Average exceedance in degrees Celsius of the heatwave threshold for each event
• extreme_high_temp: Annual average number of days in which 35°C is exceeded

Time period: 1960s through 2020s

When users ask questions, follow these steps:
1. Identify the correct metric they're asking about
2. Determine the relevant decade
3. Properly identify the country name
4. Use the tool to fetch the data and specify if you want the mean, max, or min value

Response format:
Use natural language that matches the user's question style while including these key details:
- The statistical measure used (mean/max/min)
- The specific metric analyzed
- The country name
- The decade
- The numerical result

Example:
"Based on the data, the longest heatwave duration recorded in Brazil during the 2010s was 12.3 days"
"""

graph = create_react_agent(
    tools=tools,
    model=llm,
    state_modifier=system_prompt,
)

#### Examples of expected use cases

In [6]:
question = "How long was the longest heatwave in Uruguay in the 2020s?"
ask_and_log_question(graph, question)

================================ Human Message =================================

How long was the longest heatwave in Uruguay in the 2020s?
================================== Ai Message ==================================
Tool Calls:
  get_heatwave_metric_for_country (call_jCJvezE8BwOPJfqQhU7xsWWA)
 Call ID: call_jCJvezE8BwOPJfqQhU7xsWWA
  Args:
    metric: duration
    decade: 2020s
    country: Uruguay
    reducer: max
================================= Tool Message =================================
Name: get_heatwave_metric_for_country

53.25
================================== Ai Message ==================================

Based on the data, the longest heatwave duration recorded in Uruguay during the 2020s was 53.25 days.
================================== Response Metadata ==================================
Input tokens: 2218
Output tokens: 132
Total tokens: 2350
Time taken: 4.405497074127197 seconds


In [7]:
question = "What was the average duration of heatwaves in Uy in the 1990s?"
ask_and_log_question(graph, question)

================================ Human Message =================================

What was the average duration of heatwaves in Uy in the 1990s?
================================== Ai Message ==================================
Tool Calls:
  get_heatwave_metric_for_country (call_uWs1wrllEjPF2Tzrata2C8Nq)
 Call ID: call_uWs1wrllEjPF2Tzrata2C8Nq
  Args:
    metric: duration
    decade: 1990s
    country: Uy
    reducer: mean
Country name 'Uy' standarized to 'Uruguay'
================================= Tool Message =================================
Name: get_heatwave_metric_for_country

19.611
================================== Ai Message ==================================

Based on the data, the average duration of heatwaves recorded in Uy during the 1990s was 19.6 days.
================================== Response Metadata ==================================
Input tokens: 2221
Output tokens: 130
Total tokens: 2351
Time taken: 2.2867181301116943 seconds


In [8]:
# note the country typo
question = "How many days in the 1990s was the temperature above 35°C in Urguay?"
ask_and_log_question(graph, question)

================================ Human Message =================================

How many days in the 1990s was the temperature above 35°C in Urguay?
================================== Ai Message ==================================
Tool Calls:
  get_heatwave_metric_for_country (call_4K5RuJQVCf8YvAOSWTIKS2rM)
 Call ID: call_4K5RuJQVCf8YvAOSWTIKS2rM
  Args:
    metric: extreme_high_temp
    decade: 1990s
    country: Uruguay
    reducer: mean
================================= Tool Message =================================
Name: get_heatwave_metric_for_country

1.684
================================== Ai Message ==================================

Based on the data, the annual average number of days in which the temperature exceeded 35°C in Uruguay during the 1990s was approximately 1.7 days.
================================== Response Metadata ==================================
Input tokens: 2241
Output tokens: 150
Total tokens: 2391
Time taken: 2.6211888790130615 seconds


#### Examples of not-expected use cases

In [9]:
# there is no information about the average temperatures in the dataset
question = "What is the average temperature in Uruguay?"
ask_and_log_question(graph, question)

================================ Human Message =================================

What is the average temperature in Uruguay?
================================== Ai Message ==================================

I'm sorry, but I can only provide information related to heatwave metrics such as frequency, duration, severity, and extreme high temperatures for specific countries and decades. If you have a question regarding heatwaves in Uruguay, please let me know!
================================== Response Metadata ==================================
Input tokens: 534
Output tokens: 49
Total tokens: 583
Time taken: 0.9661378860473633 seconds


In [10]:
# there is no information about the average temperatures in the dataset
question = "What was the average temperature in Uruguay in the 1990s?"
ask_and_log_question(graph, question)

================================ Human Message =================================

What was the average temperature in Uruguay in the 1990s?
================================== Ai Message ==================================

I'm sorry, but I can only provide data on heatwave metrics such as frequency, duration, severity, and extreme high temperatures, not average temperatures. If you have a specific heatwave metric in mind for Uruguay in the 1990s, please let me know!
================================== Response Metadata ==================================
Input tokens: 540
Output tokens: 55
Total tokens: 595
Time taken: 1.1022791862487793 seconds


In [11]:
# the tool only supports countries, not cities, so the LLM uses the country of the city
question = "What was the average duration of heatwaves in Montevideo in the 1990s?"
ask_and_log_question(graph, question)

================================ Human Message =================================

What was the average duration of heatwaves in Montevideo in the 1990s?
================================== Ai Message ==================================
Tool Calls:
  get_heatwave_metric_for_country (call_ffmbDbfJMUCAhQODsbI3u4hU)
 Call ID: call_ffmbDbfJMUCAhQODsbI3u4hU
  Args:
    metric: duration
    decade: 1990s
    country: Uruguay
    reducer: mean
================================= Tool Message =================================
Name: get_heatwave_metric_for_country

19.611
================================== Ai Message ==================================

Based on the data, the average duration of heatwaves recorded in Uruguay during the 1990s was approximately 19.6 days.
================================== Response Metadata ==================================
Input tokens: 2226
Output tokens: 134
Total tokens: 2360
Time taken: 2.1348989009857178 seconds


In [12]:
# in this case "Malvin" is a neighborhood of Montevideo, Uruguay,
# but the LLM incorrectly assumes it's referring to the Malvinas/Falkland Islands
question = "What was the average duration of heatwaves in Malvin in the 1990s?"
ask_and_log_question(graph, question)


================================ Human Message =================================

What was the average duration of heatwaves in Malvin in the 1990s?
================================== Ai Message ==================================
Tool Calls:
  get_heatwave_metric_for_country (call_id259naq4oEHQFuY5qNW8iDH)
 Call ID: call_id259naq4oEHQFuY5qNW8iDH
  Args:
    metric: duration
    decade: 1990s
    country: Malvin
    reducer: mean
================================= Tool Message =================================
Name: get_heatwave_metric_for_country

Error: TypeError("type NoneType doesn't define __round__ method")
 Please fix your mistakes.
================================== Ai Message ==================================

It seems there was an error in retrieving the data for Malvin. However, I believe you might be referring to the Falkland Islands, which are sometimes called Malvinas. Would you like me to check the average duration of heatwaves in the Falkland Islands during the 1990s ins